<a href="https://colab.research.google.com/github/SeungHwaJung/Auto-Mobile-/blob/main/0731_python_ensemble.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q onnxruntime opencv-python ultralytics

from ultralytics import YOLO
import onnxruntime as ort
import cv2
import numpy as np
import os
from IPython.display import Video, display

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 112.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 118.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 95.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 93.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4

In [2]:
class PeopleNet:
    def __init__(self, model_path="resnet34_peoplenet_int8.onnx"):
        self.session = ort.InferenceSession(model_path, providers=["CPUExecutionProvider"])
        self.input_name = self.session.get_inputs()[0].name
        self.input_shape = self.session.get_inputs()[0].shape

    def preprocess(self, frame):
        img = cv2.resize(frame, (960, 544))
        img = img.astype(np.float32) / 255.0
        img = img.transpose(2, 0, 1)[None, ...]
        return img

    def postprocess(self, output, frame_shape):
        h, w = frame_shape
        detections = []
        for det in output[0]:
            conf = det[4]
            if conf < 0.5: continue
            class_id = 0  # person
            xmin, ymin, xmax, ymax = map(int, det[0:4])
            detections.append([xmin, ymin, xmax, ymax, conf, class_id])
        return detections

    def infer(self, frame):
        input_tensor = self.preprocess(frame)
        outputs = self.session.run(None, {self.input_name: input_tensor})
        return self.postprocess(outputs, frame.shape[:2])

In [3]:
from ultralytics import YOLO
import os

# 1. 일반 COCO 객체 탐지 모델 (자동 다운로드)
yolo_general = YOLO("yolov8n.pt")  # 또는 yolov8s.pt, yolov8m.pt 등

# 2. 교통 객체 탐지용 커스텀 모델
# yolov8_traffic.pt 파일이 Colab 내에 없으면 업로드하거나 다운로드
custom_model_path = "yolov8_traffic.pt"

100%|██████████| 6.25M/6.25M [00:00<00:00, 112MB/s]


In [4]:
coco_names = list(yolo_general.names.values())

def map_class_id(name, source):
    if source == "peoplenet":
        return 0  # (예시용 - 실제 사용 X)
    elif source == "traffic":
        return 1  # traffic = 1
    else:
        return 2 + coco_names.index(name)  # COCO = 2~81

In [5]:
import cv2

video_path = "input_video.mp4"  # 업로드 또는 Colab 경로로 변경 가능
cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

out = cv2.VideoWriter("output_yolo8.mp4", cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))